# Notebook Overview — Run Baseline VideoQA

## Purpose

This notebook performs development-subset baseline Video Question Answering (VideoQA) experiments for the NExT-QA benchmark dataset using the Qwen2-VL-7B multimodal foundation model. The workflow is used to validate the baseline inference pipeline, evaluate parameter settings, and establish a reference configuration for subsequent pretrained-representation and autoencoder-representation experiments.

Development-subset experiments enable rapid iteration and parameter optimization while minimizing computational cost. Optimized configurations identified during these experiments are later applied to full-dataset execution within the final experiment workflow.

## Inputs

* Prepared NExT-QA video dataset

* Evidence metadata generated during video preprocessing

* NExT-QA question-answer annotation files

* NExT-QA metadata resources

* Project configuration settings

* Shared utility modules and helper functions

## Outputs

* Development-subset baseline prediction dataset
* Predicted answers
* Ground-truth answers
* Question and video metadata
* Evidence usage statistics
* Inference timing metrics
* Baseline experiment summary report
* Sample prediction results for verification

## Processing Workflow

The notebook begins by configuring the project environment, loading required configuration settings, and restoring the prepared NExT-QA video dataset. NExT-QA question-answer annotations, video inventory information, and evidence metadata are then loaded and validated. Baseline inference parameters are configured, the runtime environment and GPU resources are verified, and the Qwen2-VL-7B multimodal model and processor are initialized. An evaluation dataset is prepared from the selected NExT-QA split, after which baseline VideoQA inference is performed using sampled video evidence supplied directly to the model. Generated predictions, ground-truth answers, evidence usage statistics, and inference timing information are collected and validated before being saved to persistent storage. Finally, summary statistics and experiment reports are generated, and representative prediction samples are displayed for qualitative review and verification.


## Notes

This notebook performs direct VideoQA inference using sampled video evidence supplied directly to the Qwen2-VL-7B model. The resulting predictions establish a baseline reference for subsequent pretrained-representation and autoencoder-representation experiments. No learned representation selection or representation-learning mechanisms are applied during inference.


### 🔷 Step 1 — Clone Required Repository Files

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files and directories are available for subsequent notebook execution.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

# ============================================================
# Runtime Settings
# ============================================================
VERBOSE = True
REQUIRE_L4_GPU = True

import os
from google.colab import userdata

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------

%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    if VERBOSE:
        print("Cloning required repository directories...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    %cd {REPO_DIR}

    !git sparse-checkout init --cone

    !git sparse-checkout set \
        src \
        datasets \
        outputs

    !git checkout --quiet main

else:

    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")

    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Setup
# ------------------------------------------------------------

required_paths = [
    "src",
    "datasets",
    "outputs",
    "datasets/NExT-QA",
    "datasets/NExT-QA/questions",
    "datasets/NExT-QA/metadata",
    "src/videoqa_representation_config.py",
    "src/nextqa_video_cache.py",
    "src/nextqa_metadata.py",
    "src/video_evidence.py",
    "src/evidence_validation.py",
    "src/evidence_io.py",
]

for path in required_paths:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required path not found: {path}"
        )

print("Repository setup complete.")

if VERBOSE:
    print(f"\nCurrent directory: {os.getcwd()}")
    print("\nRepository directories:")
    !find src datasets outputs \
        -maxdepth 2 \
        -type d \
        ! -path "*/__pycache__*" | sort



### 🔷 Step 2 — Load Configuration and Initialize Paths

* Import required Python libraries and reusable project modules.
* Load centralized configuration settings from `videoqa_representation_config.py`.
* Initialize notebook input and output paths.
* Create required output directories if they do not already exist.
* Verify required input resources before continuing.



In [ ]:
# ============================================================
# Step 2: Load Configuration and Initialize Paths
# ============================================================

from pathlib import Path

import pandas as pd

from src.videoqa_representation_config import *

from src.nextqa_video_cache import *
from src.nextqa_metadata import *
from src.video_evidence import *
from src.evidence_validation import *
from src.evidence_io import *

# ------------------------------------------------------------
# Create Required Output Directories
# ------------------------------------------------------------

for output_dir in [
    EVIDENCE_METADATA_DIR,
    EVIDENCE_REPORTS_DIR,
]:
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# Verify Required Input Paths
# ------------------------------------------------------------

required_input_paths = [
    QUESTIONS_DIR,
    METADATA_DIR,
]

missing_input_paths = [
    path
    for path in required_input_paths
    if not path.exists()
]

if missing_input_paths:
    for path in missing_input_paths:
        print(f"Missing required input path: {path}")

    raise FileNotFoundError(
        "One or more required input paths are missing."
    )

print("Project configuration loaded successfully.")
print("Project utility modules loaded successfully.")
print("Input and output paths initialized successfully.")

if VERBOSE:
    print("\nLoaded Modules")
    print("-" * 60)
    print("videoqa_representation_config")
    print("nextqa_video_cache")
    print("nextqa_metadata")
    print("video_evidence")
    print("evidence_validation")
    print("evidence_io")

    print("\nInput Directories")
    print("-" * 60)
    print(f"Questions : {QUESTIONS_DIR}")
    print(f"Metadata  : {METADATA_DIR}")
    print(f"Videos    : {VIDEOS_DIR}")

    print("\nOutput Directories")
    print("-" * 60)
    print(f"Evidence Metadata : {EVIDENCE_METADATA_DIR}")
    print(f"Evidence Reports  : {EVIDENCE_REPORTS_DIR}")

    print("\nOutput Files")
    print("-" * 60)
    print(f"Evidence Metadata   : {EVIDENCE_METADATA_CSV}")
    print(f"Evidence Validation : {EVIDENCE_VALIDATION_CSV}")
    print(f"Evidence Summary    : {EVIDENCE_SUMMARY_CSV}")



### 🔷 Step 3 — Restore Local NExT-QA Video Cache

* Verify whether the NExT-QA video cache is already available in local Colab storage.
* Mount Google Drive and locate the NExT-QA release resources when local videos are missing.
* Prefer the combined NExT-QA release archive when available:
  * `releases/NExTVideo_combined.zip`
* Copy the combined archive to the local dataset archive workspace only when needed.
* Validate the local combined archive before extraction.
* Fall back to the legacy multipart archive workflow only when the combined archive is unavailable.
* Extract the NExT-QA video archive into local Colab storage when the video cache is missing.
* Verify that the restored video cache contains the expected NExT-QA video files and folder structure.
* Prepare the local video dataset for evidence generation in subsequent steps.


In [ ]:
# ============================================================
# Step 3: Restore Local NExT-QA Video Cache
# ============================================================

import shutil
import time

from google.colab import drive

# ------------------------------------------------------------
# Expected Video Cache Size
# ------------------------------------------------------------

EXPECTED_NEXTQA_VIDEO_COUNT = 5440

# ------------------------------------------------------------
# Check Existing Local Video Cache
# ------------------------------------------------------------

existing_video_files = sorted(
    VIDEOS_DIR.rglob("*.mp4")
)

if len(existing_video_files) == EXPECTED_NEXTQA_VIDEO_COUNT:

    video_cache_restore_summary = {
        "cache_status": "already_available",
        "video_count": len(existing_video_files),
        "local_videos_dir": str(VIDEOS_DIR),
        "archive_mode": "not_required",
        "verified": True,
    }

    print("Local NExT-QA video cache already available.")
    print(f"Video files found: {len(existing_video_files)}")
    print(f"Local video directory: {VIDEOS_DIR}")

else:

    print("Local NExT-QA video cache is missing or incomplete.")
    print(f"Video files found locally: {len(existing_video_files)}")
    print("Restoring video cache from Google Drive release archive...")

    # --------------------------------------------------------
    # Mount Google Drive
    # --------------------------------------------------------

    GOOGLE_DRIVE_MOUNT = "/content/drive"

    if not os.path.exists(GOOGLE_DRIVE_MOUNT):

        if VERBOSE:
            print("\nMounting Google Drive...")

        drive.mount(GOOGLE_DRIVE_MOUNT)

    else:

        if VERBOSE:
            print("\nGoogle Drive is already mounted.")

    drive_root = Path(GOOGLE_DRIVE_MOUNT) / "MyDrive"

    if not drive_root.exists():

        raise FileNotFoundError(
            "Unable to access Google Drive root directory."
        )

    # --------------------------------------------------------
    # Configure Google Drive Release and Local Cache Paths
    # --------------------------------------------------------

    DRIVE_DATASET_DIR = drive_root / "VideoQA_Project" / "NExT-QA"
    DRIVE_RELEASES_DIR = DRIVE_DATASET_DIR / "releases"

    LOCAL_ARCHIVE_DIR = DATASET_DIR / "archives"
    LOCAL_ARCHIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    COMBINED_ARCHIVE_NAME = "NExTVideo_combined.zip"

    DRIVE_COMBINED_ARCHIVE_PATH = (
        DRIVE_RELEASES_DIR /
        COMBINED_ARCHIVE_NAME
    )

    COMBINED_ARCHIVE_PATH = (
        LOCAL_ARCHIVE_DIR /
        COMBINED_ARCHIVE_NAME
    )

    required_archive_files = [
        "NExTVideo.z01",
        "NExTVideo.z02",
        "NExTVideo.z03",
        "NExTVideo.z04",
        "NExTVideo.z05",
        "NExTVideo.z06",
        "NExTVideo.zip",
    ]

    if not DRIVE_RELEASES_DIR.exists():

        raise FileNotFoundError(
            "Google Drive NExT-QA releases directory not found:\n"
            f"{DRIVE_RELEASES_DIR}"
        )

    # --------------------------------------------------------
    # Prefer Combined Archive
    # --------------------------------------------------------

    if DRIVE_COMBINED_ARCHIVE_PATH.exists():

        print("\nPreferred combined NExT-QA archive found.")
        print(f"Source archive : {DRIVE_COMBINED_ARCHIVE_PATH}")
        print(f"Local archive  : {COMBINED_ARCHIVE_PATH}")

        source_size = DRIVE_COMBINED_ARCHIVE_PATH.stat().st_size

        copy_start_time = time.time()

        if COMBINED_ARCHIVE_PATH.exists():

            local_size = COMBINED_ARCHIVE_PATH.stat().st_size

            if local_size == source_size:

                print("Local combined archive already exists with matching size.")
                print("Archive copy skipped.")

            else:

                print(
                    "Local combined archive exists but size differs. "
                    "Replacing local archive."
                )

                COMBINED_ARCHIVE_PATH.unlink()

                shutil.copy2(
                    DRIVE_COMBINED_ARCHIVE_PATH,
                    COMBINED_ARCHIVE_PATH,
                )

        else:

            print("Copying preferred combined archive to local storage...")

            shutil.copy2(
                DRIVE_COMBINED_ARCHIVE_PATH,
                COMBINED_ARCHIVE_PATH,
            )

        copy_elapsed_time = time.time() - copy_start_time

        local_size = COMBINED_ARCHIVE_PATH.stat().st_size

        if local_size != source_size:

            raise ValueError(
                "Combined archive copy failed size verification."
            )

        archive_restore_summary = {
            "archive_mode": "combined",
            "source_archive": str(DRIVE_COMBINED_ARCHIVE_PATH),
            "local_archive": str(COMBINED_ARCHIVE_PATH),
            "archive_size_gb": local_size / (1024 ** 3),
            "copy_elapsed_seconds": copy_elapsed_time,
            "verified": True,
        }

        print("Combined archive copied and verified.")
        print(f"Elapsed time : {copy_elapsed_time:.1f} seconds")
        print(f"Local size   : {local_size / (1024 ** 3):.2f} GB")

    # --------------------------------------------------------
    # Fallback: Legacy Multipart Archive Workflow
    # --------------------------------------------------------

    else:

        print(
            "\nPreferred combined archive not found. "
            "Falling back to legacy multipart archive workflow."
        )

        archive_verification_summary = verify_nextqa_archive_parts(
            archive_parts_dir=DRIVE_RELEASES_DIR,
            required_archive_files=required_archive_files,
            verbose=VERBOSE,
        )

        local_archive_summary = copy_nextqa_archive_parts_to_local(
            source_archive_dir=DRIVE_RELEASES_DIR,
            local_archive_dir=LOCAL_ARCHIVE_DIR,
            required_archive_files=required_archive_files,
            verbose=VERBOSE,
        )

        archive_restore_summary = build_combined_nextqa_archive(
            local_archive_dir=LOCAL_ARCHIVE_DIR,
            combined_archive_path=COMBINED_ARCHIVE_PATH,
            split_archive_name="NExTVideo.zip",
            required_archive_files=required_archive_files,
            force_rebuild=True,
            verbose=VERBOSE,
        )

    # --------------------------------------------------------
    # Extract Combined Archive and Verify Video Cache
    # --------------------------------------------------------

    print("\nExtracting or verifying local NExT-QA video cache...")

    extraction_start_time = time.time()

    extract_summary = extract_nextqa_video_archive(
        combined_archive_path=COMBINED_ARCHIVE_PATH,
        local_videos_dir=VIDEOS_DIR,
        force_extract=False,
        verbose=VERBOSE,
    )

    extraction_elapsed_time = time.time() - extraction_start_time

    restored_video_files = sorted(
        VIDEOS_DIR.rglob("*.mp4")
    )

    if len(restored_video_files) != EXPECTED_NEXTQA_VIDEO_COUNT:

        raise ValueError(
            "NExT-QA video cache verification failed. "
            f"Expected {EXPECTED_NEXTQA_VIDEO_COUNT} videos, "
            f"found {len(restored_video_files)}."
        )

    video_cache_restore_summary = {
        "cache_status": "restored",
        "video_count": len(restored_video_files),
        "local_videos_dir": str(VIDEOS_DIR),
        "archive_mode": archive_restore_summary.get(
            "archive_mode",
            "unknown",
        ),
        "archive_restore_summary": archive_restore_summary,
        "extract_summary": extract_summary,
        "extraction_elapsed_seconds": extraction_elapsed_time,
        "verified": True,
    }

    print("\nLocal NExT-QA video cache restored and verified.")
    print(f"Video files found : {len(restored_video_files)}")
    print(f"Elapsed time      : {extraction_elapsed_time:.1f} seconds")
    print(f"Local videos      : {VIDEOS_DIR}")

print("\nLocal NExT-QA video cache is ready.")


### 🔷 Step 4 — Load NExT-QA Metadata and Video Inventory

* Load NExT-QA annotation files for the train, validation, and test splits.
* Combine annotation records into a unified evaluation dataset.
* Build a video inventory from available video files.
* Associate video inventory information with annotation records.


In [ ]:
# ============================================================
# Step 4: Load NExT-QA Metadata and Video Inventory
# ============================================================

# ------------------------------------------------------------
# Load NExT-QA Annotation Files
# ------------------------------------------------------------

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Build Local Video Inventory
# ------------------------------------------------------------

video_inventory_df = build_nextqa_video_inventory(
    videos_dir=VIDEOS_DIR,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Attach Video Inventory Information
# ------------------------------------------------------------

annotations_with_videos_df = (
    attach_video_inventory_to_annotations(
        annotations=annotations_df,
        video_inventory=video_inventory_df,
        verbose=VERBOSE,
    )
)

# ------------------------------------------------------------
# Generate Split Summary
# ------------------------------------------------------------

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

# ------------------------------------------------------------
# Verify Annotation Coverage
# ------------------------------------------------------------

coverage_summary = verify_annotation_video_coverage(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nNExT-QA metadata and video inventory loaded successfully.")

print(f"Annotation records : {len(annotations_df):,}")
print(f"Video inventory    : {len(video_inventory_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)



### 🔷 Step 5 — Load Evidence Metadata

* Load evidence metadata generated during Notebook 02 preprocessing.
* Verify that evidence records are available for all required videos.
* Prepare evidence metadata structures used during frame selection and inference.
* Compute and display evidence repository statistics.


In [ ]:
# ============================================================
# Step 5: Load Evidence Metadata
# ============================================================

import pandas as pd
from pathlib import Path

EVIDENCE_METADATA_FILE = (
    Path(REPO_DIR)
    / "outputs"
    / "evidence"
    / "metadata"
    / "evidence_metadata.csv"
)

print("Loading evidence metadata...")

if not EVIDENCE_METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Evidence metadata file not found:\n{EVIDENCE_METADATA_FILE}"
    )

evidence_df = pd.read_csv(EVIDENCE_METADATA_FILE)

print("Evidence metadata loaded successfully.")
print(f"Evidence records: {len(evidence_df):,}")
print(f"Columns:          {len(evidence_df.columns)}")

print("\nEvidence Metadata Preview:")
display(evidence_df.head())

print("\nEvidence Metadata Summary:")
display(
    pd.DataFrame({
        "Metric": [
            "Total Evidence Records",
            "Unique Videos"
        ],
        "Value": [
            len(evidence_df),
            evidence_df["video_id"].nunique()
            if "video_id" in evidence_df.columns
            else "N/A"
        ]
    })
)



### 🔷 Step 6 — Define Development-Subset Inference Parameters

* Configure baseline VideoQA evaluation settings and experiment controls.
* Define development-subset limits and sampling parameters.
* Configure frame selection and evidence usage settings.
* Define model generation parameters used during inference.


In [ ]:
# ============================================================
# Step 6: Define Development-Subset Inference Parameters
# ============================================================

BASELINE_CONFIG = {
    # Evaluation control
    "evaluation_split": "val",
    "development_subset_size": 25,
    "random_seed": 42,

    # Development-subset experiments are used for
    # parameter optimization and workflow validation.

    # Video evidence selection
    "max_evidence_records_per_video": 5,
    "max_frames_per_question": 8,

    # Model generation settings
    "max_new_tokens": 64,
    "temperature": 0.0,
    "do_sample": False,

    # Output control
    "save_intermediate_results": True,
    "verbose": True,
}

print("\nBaseline Configuration:")
for key, value in BASELINE_CONFIG.items():
    print(f"  {key:<32}: {value}")



### 🔷 Step 7 — Verify GPU Runtime and Model Dependencies

* Verify that the Colab runtime satisfies Qwen2-VL-7B execution requirements.
* Confirm PyTorch installation and CUDA availability.
* Display GPU hardware information and runtime configuration.
* Verify required model and processor dependencies.


In [ ]:
# ============================================================
# Step 7: Verify GPU Runtime and Model Dependencies
# ============================================================

import sys
import platform
import importlib

print("Verifying GPU runtime and model dependencies...\n")

# ------------------------------------------------------------
# Runtime Information
# ------------------------------------------------------------

print("Runtime Information")
print("-" * 60)
print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch / CUDA Verification
# ------------------------------------------------------------

try:
    import torch

    print("\nPyTorch Information")
    print("-" * 60)
    print(f"PyTorch Version : {torch.__version__}")
    print(f"CUDA Available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():

        print(f"CUDA Version    : {torch.version.cuda}")
        print(f"GPU Count       : {torch.cuda.device_count()}")

        for idx in range(torch.cuda.device_count()):

            gpu_name = torch.cuda.get_device_name(idx)

            gpu_props = torch.cuda.get_device_properties(idx)

            total_memory_gb = (
                gpu_props.total_memory /
                (1024 ** 3)
            )

            print(
                f"GPU {idx}          : "
                f"{gpu_name}"
            )

            print(
                f"GPU {idx} Memory   : "
                f"{total_memory_gb:.1f} GB"
            )

        allocated_gb = (
            torch.cuda.memory_allocated() /
            (1024 ** 3)
        )

        reserved_gb = (
            torch.cuda.memory_reserved() /
            (1024 ** 3)
        )

        print(
            f"Allocated Memory : "
            f"{allocated_gb:.2f} GB"
        )

        print(
            f"Reserved Memory  : "
            f"{reserved_gb:.2f} GB"
        )

        primary_gpu = torch.cuda.get_device_name(0)

        if REQUIRE_L4_GPU and "L4" not in primary_gpu:
            raise RuntimeError(
                f"Required NVIDIA L4 GPU not available. "
                f"Detected GPU: {primary_gpu}. "
                "Change the Colab runtime to L4 before continuing."
            )

        if "T4" in primary_gpu:

            print(
                "\nWARNING: NVIDIA T4 GPU detected "
                "(approximately 16 GB VRAM)."
            )

            print(
                "Large multimodal inference workloads "
                "may require reduced frame counts or "
                "memory optimization settings."
            )

        elif "L4" in primary_gpu:

            print(
                "\nNVIDIA L4 GPU detected "
                "(approximately 24 GB VRAM)."
            )

        device = "cuda"

    else:

        print("WARNING: No CUDA GPU detected.")
        device = "cpu"

except Exception as e:

    print(f"ERROR: Unable to load PyTorch ({e})")
    device = "cpu"

# ------------------------------------------------------------
# Required Packages
# ------------------------------------------------------------

required_packages = [
    "transformers",
    "accelerate",
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "PIL",
]

print("\nDependency Verification")
print("-" * 60)

dependency_status = []

for package_name in required_packages:

    try:
        module = importlib.import_module(package_name)

        version = getattr(module, "__version__", "unknown")

        dependency_status.append(
            {
                "package": package_name,
                "status": "OK",
                "version": version,
            }
        )

        print(f"[OK]   {package_name:<15} {version}")

    except Exception:
        dependency_status.append(
            {
                "package": package_name,
                "status": "MISSING",
                "version": "",
            }
        )

        print(f"[FAIL] {package_name}")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

missing_packages = [
    item["package"]
    for item in dependency_status
    if item["status"] != "OK"
]

print("\nVerification Summary")
print("-" * 60)

print(f"Execution Device : {device}")

if len(missing_packages) == 0:
    print("All required dependencies are available.")
else:
    print("Missing packages:")
    for pkg in missing_packages:
        print(f"  - {pkg}")



### 🔷 Step 8 — Load Qwen2-VL-7B Model and Processor

* Load the Qwen2-VL-7B multimodal model used for baseline VideoQA inference.
* Load the associated processor for multimodal input preparation.
* Configure model execution on available GPU resources.
* Verify successful model and processor initialization.


In [ ]:
# ============================================================
# Step 8: Load Qwen2-VL-7B Model and Processor
# ============================================================

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("Loading Qwen2-VL-7B model and processor...")

MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for Qwen2-VL-7B inference. "
        "Please switch Colab runtime to GPU."
    )

device = "cuda"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Qwen2-VL-7B model and processor loaded successfully.")
print(f"Model ID : {MODEL_ID}")
print(f"Device   : {device}")
print(f"Dtype    : {model.dtype}")



### 🔷 Step 9 — Prepare Development Evaluation Subset

* Select the configured NExT-QA evaluation split.
* Apply optional sampling limits to control experiment size and runtime.
* Validate required question, answer, and video metadata fields.
* Prepare the development evaluation subset used for parameter optimization.


In [ ]:
# ============================================================
# Step 9: Prepare Evaluation Dataset
# ============================================================

import random
import pandas as pd

print("Preparing development evaluation subset...")

evaluation_split = BASELINE_CONFIG["evaluation_split"]
development_subset_size = BASELINE_CONFIG["development_subset_size"]
random_seed = BASELINE_CONFIG["random_seed"]

if "split" not in annotations_df.columns:
    raise ValueError("annotations_df must contain a 'split' column.")

eval_df = annotations_df[
    annotations_df["split"] == evaluation_split
].copy()

if len(eval_df) == 0:
    raise ValueError(f"No records found for split: {evaluation_split}")

print(f"Development subset size: {development_subset_size:,}")

sample_size = min(development_subset_size, len(eval_df))

eval_df = eval_df.sample(
    n=sample_size,
    random_state=random_seed
).reset_index(drop=True)

print(f"Selected evaluation samples: {len(eval_df):,}")

# ------------------------------------------------------------
# Attach video file paths
# ------------------------------------------------------------

if "video" not in eval_df.columns:
    raise ValueError("annotations_df must contain a 'video' column.")

VIDEO_DIR = (
    Path(REPO_DIR)
    / "datasets"
    / "NExT-QA"
    / "videos"
)

def resolve_video_path(video_id):
    matches = list(VIDEO_DIR.rglob(f"{video_id}.mp4"))

    if len(matches) == 0:
        return None

    return matches[0]

eval_df["video_path"] = eval_df["video"].apply(resolve_video_path)

missing_video_count = eval_df["video_path"].isna().sum()

print(f"Missing video files: {missing_video_count}")

if missing_video_count > 0:
    display(eval_df[eval_df["video_path"].isna()].head())
    raise FileNotFoundError(
        "One or more evaluation samples do not have matching video files."
    )

# ------------------------------------------------------------
# Attach evidence metadata summary
# ------------------------------------------------------------

if "video_id" in evidence_df.columns:
    evidence_video_col = "video_id"
elif "video" in evidence_df.columns:
    evidence_video_col = "video"
else:
    raise ValueError(
        "evidence_df must contain either 'video_id' or 'video'."
    )

evidence_counts = (
    evidence_df
    .groupby(evidence_video_col)
    .size()
    .reset_index(name="evidence_record_count")
)

eval_df = eval_df.merge(
    evidence_counts,
    left_on="video",
    right_on=evidence_video_col,
    how="left"
)

eval_df["evidence_record_count"] = (
    eval_df["evidence_record_count"]
    .fillna(0)
    .astype(int)
)

def answer_index_to_text(row):

    answer_idx = int(row["answer"])

    option_col = f"a{answer_idx}"

    return row[option_col]

eval_df["ground_truth_text"] = (
    eval_df.apply(
        answer_index_to_text,
        axis=1
    )
)



missing_evidence_count = (eval_df["evidence_record_count"] == 0).sum()

print(f"Samples without evidence metadata: {missing_evidence_count}")

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

required_eval_columns = [
    "video",
    "question",
    "answer",
    "video_path",
    "evidence_record_count",
]

missing_columns = [
    col for col in required_eval_columns
    if col not in eval_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required evaluation columns: {missing_columns}")

print("\nEvaluation dataset prepared successfully.")
print(f"Evaluation samples : {len(eval_df):,}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Random seed        : {random_seed}")

print("\nEvaluation Dataset Preview:")
display(eval_df.head())



### 🔷 Step 10 — Run Development-Subset Baseline VideoQA Inference

* Sample representative video frames from each evaluation video.
* Construct multimodal prompts consisting of video evidence and associated questions.
* Execute development-subset baseline VideoQA inference using Qwen2-VL-7B using sampled video evidence supplied directly to the model.
* Generate predicted answers and record evidence usage statistics.


In [ ]:
# ============================================================
# Step 10: Run Development-Subset Baseline VideoQA Inference
# ============================================================

import time
import gc
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image
import cv2
import torch

# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------

def clear_gpu_memory():
    """
    Release unused Python and CUDA memory between inference samples.
    """

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def sample_video_frames(
    video_path,
    num_frames=8
):
    """
    Uniformly sample frames from a video.
    """

    cap = cv2.VideoCapture(str(video_path))

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return []

    frame_indices = [
        int(i * frame_count / num_frames)
        for i in range(num_frames)
    ]

    frames = []

    for idx in frame_indices:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)

        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frames.append(
                Image.fromarray(frame)
            )

    cap.release()

    return frames


def run_videoqa_inference(
    video_path,
    question
):
    """
    Execute baseline Qwen2-VL inference.
    """

    frames = None
    messages = None
    text = None
    inputs = None
    generated_ids = None
    generated_ids_trimmed = None
    output_text = None

    try:

        frames = sample_video_frames(
            video_path,
            BASELINE_CONFIG["max_frames_per_question"]
        )

        if len(frames) == 0:
            return "VIDEO_READ_ERROR"

        messages = [
            {
                "role": "user",
                "content": (
                    [{"type": "image", "image": frame}
                     for frame in frames]
                    +
                    [{
                        "type": "text",
                        "text": (
                            "Answer the following video question "
                            "as concisely as possible.\n\n"
                            f"Question: {question}"
                        )
                    }]
                )
            }
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = processor(
            text=[text],
            images=frames,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        generate_kwargs = {
            "max_new_tokens": BASELINE_CONFIG["max_new_tokens"],
            "do_sample": BASELINE_CONFIG["do_sample"],
        }

        if BASELINE_CONFIG["do_sample"]:
            generate_kwargs["temperature"] = BASELINE_CONFIG["temperature"]

        with torch.no_grad():

            generated_ids = model.generate(
                **inputs,
                **generate_kwargs
            )

        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(
                inputs["input_ids"],
                generated_ids
            )
        ]

        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

        return output_text.strip()

    finally:

        del frames
        del messages
        del text
        del inputs
        del generated_ids
        del generated_ids_trimmed
        del output_text

        clear_gpu_memory()


# ------------------------------------------------------------
# Baseline Inference Loop
# ------------------------------------------------------------

print(
    f"Running development-subset baseline inference "
    f"on {len(eval_df):,} samples..."
)

results = []

clear_gpu_memory()

start_time = time.time()

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Running Baseline VideoQA"
):

    try:

        prediction = run_videoqa_inference(
            row["video_path"],
            row["question"]
        )

    except Exception as e:

        prediction = f"ERROR: {str(e)}"
        clear_gpu_memory()

    results.append({
        "video": row["video"],
        "question": row["question"],
        "ground_truth": row["ground_truth_text"],
        "prediction": prediction,
        "evidence_record_count": row["evidence_record_count"]
    })

elapsed_time = time.time() - start_time

prediction_df = pd.DataFrame(results)

print(f"Evaluation samples : {len(prediction_df):,}")
print(f"Elapsed time       : {elapsed_time:.1f} seconds")
print(
    f"Average/sample     : "
    f"{elapsed_time / len(prediction_df):.2f} seconds"
)

display(prediction_df.head())



### 🔷 Step 11 — Validate Prediction Results

* Verify that baseline prediction records were generated successfully.
* Validate required prediction fields and output structure.
* Check for missing or invalid prediction values.
* Confirm dataset integrity before saving results.


In [ ]:
# ============================================================
# Step 11: Validate Prediction Results
# ============================================================

import pandas as pd

print("Validating baseline prediction results...")

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 10 first.")

required_prediction_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
    "evidence_record_count",
]

missing_columns = [
    col for col in required_prediction_columns
    if col not in prediction_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required prediction columns: {missing_columns}")

validation_summary = {
    "total_predictions": len(prediction_df),
    "missing_predictions": prediction_df["prediction"].isna().sum(),
    "empty_predictions": (
        prediction_df["prediction"].astype(str).str.strip() == ""
    ).sum(),
    "error_predictions": (
        prediction_df["prediction"].astype(str).str.startswith("ERROR")
    ).sum(),
    "video_read_errors": (
        prediction_df["prediction"].astype(str) == "VIDEO_READ_ERROR"
    ).sum(),
    "unique_videos": prediction_df["video"].nunique(),
}

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Count"]
)

display(validation_df)

problem_predictions_df = prediction_df[
    prediction_df["prediction"].isna()
    | (prediction_df["prediction"].astype(str).str.strip() == "")
    | (prediction_df["prediction"].astype(str).str.startswith("ERROR"))
    | (prediction_df["prediction"].astype(str) == "VIDEO_READ_ERROR")
].copy()

if len(problem_predictions_df) > 0:
    print("\nProblem predictions detected:")
    display(problem_predictions_df)
else:
    print("\nPrediction validation passed. No missing, empty, or error predictions detected.")

# ------------------------------------------------------------
# Runtime Projection
# ------------------------------------------------------------

if "elapsed_time" in globals():
    avg_time_per_sample = elapsed_time / len(prediction_df)
    total_dataset_size = len(annotations_df)
    projected_seconds = avg_time_per_sample * total_dataset_size
    projected_hours = projected_seconds / 3600

    print("\nRuntime Projection")
    print("-" * 60)
    print(f"Average Time per Sample : {avg_time_per_sample:.2f} sec")
    print(f"Dataset Size            : {total_dataset_size:,}")
    print(f"Projected Runtime       : {projected_hours:.2f} hours")



### 🔷 Step 12 — Save Prediction Results

* Save baseline prediction records to persistent output storage.
* Create required output directories when necessary.
* Verify successful file creation and storage operations.
* Record prediction file locations for downstream evaluation workflows.


In [ ]:
# ============================================================
# Step 12: Save Prediction Results
# ============================================================

from pathlib import Path

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 11 first.")

BASELINE_OUTPUT_DIR = (
    Path(REPO_DIR)
    / "outputs"
    / "baseline"
)
BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prediction_output_file = BASELINE_OUTPUT_DIR / "baseline_predictions.csv"

prediction_df.to_csv(
    prediction_output_file,
    index=False
)

print("Baseline prediction results saved successfully.")
print(f"Prediction file : {prediction_output_file}")
print(f"Records saved   : {len(prediction_df):,}")



### 🔷 Step 13 — Generate Development-Subset Baseline Summary Report

* Compute summary statistics describing baseline VideoQA execution results.
* Aggregate prediction counts, evidence usage statistics, and runtime metrics.
* Generate experiment documentation and evaluation summaries.
* Save summary reports for later comparison with pretrained-representation and autoencoder-representation experiments.


In [ ]:
# ============================================================
# Step 13: Generate Development-Subset Baseline Summary Report
# ============================================================

import pandas as pd
from pathlib import Path

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 11 first.")

if "annotations_df" not in globals():
    raise NameError("annotations_df was not found.")

if "elapsed_time" not in globals():
    raise NameError("elapsed_time was not found.")

BASELINE_OUTPUT_DIR = (
    Path(REPO_DIR)
    / "outputs"
    / "baseline"
)

BASELINE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

summary_output_file = (
    BASELINE_OUTPUT_DIR
    / "baseline_summary.csv"
)

# ------------------------------------------------------------
# Prediction Statistics
# ------------------------------------------------------------

total_predictions = len(prediction_df)

missing_predictions = (
    prediction_df["prediction"]
    .isna()
    .sum()
)

empty_predictions = (
    prediction_df["prediction"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

error_predictions = (
    prediction_df["prediction"]
    .astype(str)
    .str.startswith("ERROR")
    .sum()
)

video_read_errors = (
    prediction_df["prediction"]
    .astype(str)
    .eq("VIDEO_READ_ERROR")
    .sum()
)

valid_predictions = (
    total_predictions
    - missing_predictions
    - empty_predictions
    - error_predictions
    - video_read_errors
)

avg_evidence_records = (
    prediction_df["evidence_record_count"]
    .mean()
)

# ------------------------------------------------------------
# Runtime Statistics
# ------------------------------------------------------------

avg_time_per_sample = (
    elapsed_time / total_predictions
)

total_dataset_size = len(annotations_df)

projected_full_dataset_seconds = (
    avg_time_per_sample
    * total_dataset_size
)

projected_full_dataset_hours = (
    projected_full_dataset_seconds
    / 3600
)

evaluation_split_size = len(
    annotations_df[
        annotations_df["split"]
        == BASELINE_CONFIG["evaluation_split"]
    ]
)

projected_eval_split_minutes = (
    (avg_time_per_sample * evaluation_split_size)
    / 60
)

# ------------------------------------------------------------
# Summary Report
# ------------------------------------------------------------

baseline_summary_df = pd.DataFrame(
    [
        {
            "metric": "total_predictions",
            "value": total_predictions,
        },
        {
            "metric": "valid_predictions",
            "value": valid_predictions,
        },
        {
            "metric": "missing_predictions",
            "value": missing_predictions,
        },
        {
            "metric": "empty_predictions",
            "value": empty_predictions,
        },
        {
            "metric": "error_predictions",
            "value": error_predictions,
        },
        {
            "metric": "video_read_errors",
            "value": video_read_errors,
        },
        {
            "metric": "unique_videos",
            "value": prediction_df["video"].nunique(),
        },
        {
            "metric": "average_evidence_records_per_sample",
            "value": round(avg_evidence_records, 2),
        },
        {
            "metric": "elapsed_time_seconds",
            "value": round(elapsed_time, 2),
        },
        {
            "metric": "average_time_per_sample_seconds",
            "value": round(avg_time_per_sample, 2),
        },
        {
            "metric": "projected_validation_runtime_minutes",
            "value": round(
                projected_eval_split_minutes,
                2
            ),
        },
        {
            "metric": "projected_full_dataset_runtime_hours",
            "value": round(
                projected_full_dataset_hours,
                2
            ),
        },
    ]
)

baseline_summary_df.to_csv(
    summary_output_file,
    index=False
)

print(
    f"Baseline summary report saved: "
    f"{summary_output_file}"
)

display(baseline_summary_df)



### 🔷 Step 14 — Display Sample Predictions

* Randomly select representative prediction records from the baseline evaluation results.
* Display questions, ground-truth answers, and model predictions.
* Review evidence usage information associated with each prediction.
* Support qualitative assessment of baseline VideoQA performance.


In [ ]:
# ============================================================
# Step 14: Display Sample Predictions
# ============================================================

import pandas as pd

if "prediction_df" not in globals():
    raise NameError("prediction_df was not found. Run Step 11 first.")

sample_count = min(10, len(prediction_df))

sample_predictions_df = (
    prediction_df
    .sample(
        n=sample_count,
        random_state=BASELINE_CONFIG["random_seed"]
    )
    .reset_index(drop=True)
)

display_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
    "evidence_record_count",
]

display(sample_predictions_df[display_columns])

